# 32 English Dataset Alignment and Processing

This notebook moves the English Kaggle resume dataset out of the pilot stage and into a processed-data stage that matches the project's `30+` notebook line.

What it does:
- downloads and loads the English dataset,
- cleans and standardizes text fields,
- maps raw English categories into the current 9 target classes,
- builds ready-to-use `train/val/test` splits,
- prepares English base eval and city-swap counterfactual artifacts,
- saves processed outputs for the next English training or evaluation round.


In [ ]:
# Install kagglehub on the fly if needed
try:
    import kagglehub
except ModuleNotFoundError:
    import sys
    !{sys.executable} -m pip install --quiet kagglehub
    import kagglehub

import json
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ModuleNotFoundError:
    HAS_MPL = False

CWD = Path.cwd()
NOTEBOOK_DIR = CWD if CWD.name == "notebooks" else (CWD / "notebooks")
REPO_ROOT = NOTEBOOK_DIR.parent
FIGURES_DIR = REPO_ROOT / "figures" / "english_dataset_alignment"
RESULTS_DIR = REPO_ROOT / "notebooks" / "results" / "english_dataset_alignment"
PROCESSED_DIR = REPO_ROOT / "data" / "processed" / "english_dataset_v1"

for directory in [FIGURES_DIR, RESULTS_DIR, PROCESSED_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Figures dir:", FIGURES_DIR)
print("Results dir:", RESULTS_DIR)
print("Processed dir:", PROCESSED_DIR)


In [ ]:
KAGGLE_HANDLE = "rayyankauchali0/resume-dataset"
download_path = Path(kagglehub.dataset_download(KAGGLE_HANDLE))
print("Downloaded to:", download_path)

def load_jsonl(path_obj: Path) -> pd.DataFrame:
    rows = []
    with path_obj.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

def load_dataset(download_dir: Path) -> pd.DataFrame:
    candidates = list(download_dir.rglob("*.csv")) + list(download_dir.rglob("*.jsonl")) + list(download_dir.rglob("*.json"))
    if not candidates:
        for archive in download_dir.rglob("*.zip"):
            with zipfile.ZipFile(archive) as zf:
                extract_dir = archive.with_suffix("")
                zf.extractall(extract_dir)
        candidates = list(download_dir.rglob("*.csv")) + list(download_dir.rglob("*.jsonl")) + list(download_dir.rglob("*.json"))

    if not candidates:
        raise FileNotFoundError(f"No CSV/JSONL/JSON files found under {download_dir}")

    ranked = sorted(candidates, key=lambda p: (p.suffix != ".csv", len(str(p))))
    data_path = ranked[0]
    print("Using data file:", data_path)

    if data_path.suffix == ".csv":
        return pd.read_csv(data_path)
    if data_path.suffix == ".jsonl":
        return load_jsonl(data_path)
    return pd.read_json(data_path)

raw_df = load_dataset(download_path)
print("Shape:", raw_df.shape)
print("Columns:", list(raw_df.columns))
raw_df.head(3)


In [ ]:
TEXT_CANDIDATES = ["Resume_Text", "resume_text", "Resume", "Text", "Summary", "Experience", "Education", "Skills"]
CATEGORY_CANDIDATES = ["Category", "category", "Job_Role", "job_role"]
LOCATION_CANDIDATES = ["Location", "location", "City", "city", "Address", "address"]
SOURCE_CANDIDATES = ["Source", "source"]

def first_present(columns, candidates):
    for col in candidates:
        if col in columns:
            return col
    return None

resume_text_col = first_present(raw_df.columns, TEXT_CANDIDATES)
category_col = first_present(raw_df.columns, CATEGORY_CANDIDATES)
location_col = first_present(raw_df.columns, LOCATION_CANDIDATES)
source_col = first_present(raw_df.columns, SOURCE_CANDIDATES)

if resume_text_col is None or category_col is None:
    raise ValueError(f"Could not identify resume text/category columns. text={resume_text_col}, category={category_col}")

df = raw_df.copy()
df["resume_text"] = df[resume_text_col].fillna("").astype(str)
df["raw_category"] = df[category_col].fillna("").astype(str).str.strip()
df["raw_location"] = df[location_col].fillna("").astype(str) if location_col else ""
df["source"] = df[source_col].fillna("unknown").astype(str) if source_col else "unknown"

df["resume_text"] = (
    df["resume_text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.replace(r"\x00", " ", regex=True)
    .str.strip()
)

df = df[df["resume_text"] != ""].copy()
df = df[df["raw_category"] != ""].copy()
df = df.drop_duplicates(subset=["resume_text", "raw_category"]).reset_index(drop=True)

df["text_len_chars"] = df["resume_text"].str.len()
df["text_len_words"] = df["resume_text"].str.split().str.len()
df = df[df["text_len_words"] >= 20].copy()

cleaning_summary = pd.DataFrame([
    {
        "rows_after_cleaning": len(df),
        "unique_raw_categories": int(df["raw_category"].nunique()),
        "median_words": float(df["text_len_words"].median()) if len(df) else 0.0,
        "empty_source_share": float((df["source"].astype(str).str.strip() == "").mean()) if len(df) else 0.0,
    }
])
cleaning_summary.to_csv(RESULTS_DIR / "32_english_cleaning_summary.csv", index=False)
cleaning_summary


In [ ]:
TARGET_9_CLASSES = [
    "backend_general_dev",
    "web_frontend",
    "sysadmin_devops_network",
    "project_product",
    "sales_account",
    "tech_support_helpdesk",
    "it_governance_leadership",
    "technical_specialized",
    "generic_it_ops",
]

CATEGORY_TO_TARGET = {
    "Java Developer": "backend_general_dev",
    "Python Developer": "backend_general_dev",
    "DotNet Developer": "backend_general_dev",
    "ETL Developer": "backend_general_dev",
    "SQL Developer": "backend_general_dev",
    "Software Developer": "backend_general_dev",
    "Backend Developer": "backend_general_dev",
    "Full Stack Developer": "backend_general_dev",
    "Blockchain Developer": "backend_general_dev",
    "Mobile Developer": "backend_general_dev",
    "React Developer": "web_frontend",
    "Frontend Developer": "web_frontend",
    "Web Designing": "web_frontend",
    "UI/UX Designer": "web_frontend",
    "Designer": "web_frontend",
    "Data Science": "technical_specialized",
    "Database": "technical_specialized",
    "AI Engineer": "technical_specialized",
    "Machine Learning Engineer": "technical_specialized",
    "Database Administrator": "technical_specialized",
    "SAP Developer": "technical_specialized",
    "Blockchain": "technical_specialized",
    "DevOps": "sysadmin_devops_network",
    "DevOps Engineer": "sysadmin_devops_network",
    "Cloud Engineer": "sysadmin_devops_network",
    "Network Security Engineer": "sysadmin_devops_network",
    "Cybersecurity Analyst": "sysadmin_devops_network",
    "Site Reliability Engineer": "sysadmin_devops_network",
    "System Administrator": "sysadmin_devops_network",
    "Business Analyst": "project_product",
    "Product Manager": "project_product",
    "PMO": "project_product",
    "Sales": "sales_account",
    "Advocate": "sales_account",
    "Testing": "tech_support_helpdesk",
    "QA Engineer": "tech_support_helpdesk",
    "Technical Lead": "it_governance_leadership",
    "Engineering Manager": "it_governance_leadership",
    "Operations Manager": "it_governance_leadership",
    "Principal Engineer": "it_governance_leadership",
    "Technical Writer": "generic_it_ops",
    "Digital Media": "generic_it_ops",
    "HR": "generic_it_ops",
    "Arts": "generic_it_ops",
}

def normalize_category(value: str):
    key = str(value).strip()
    if key in CATEGORY_TO_TARGET:
        return CATEGORY_TO_TARGET[key], "direct_map"

    lowered = key.lower()
    if any(token in lowered for token in ["frontend", "react", "ui", "ux", "web design", "designer"]):
        return "web_frontend", "keyword_frontend"
    if any(token in lowered for token in ["backend", "developer", "software", "programmer", "java", "python", "dotnet", "full stack", "mobile", "sql", "etl"]):
        return "backend_general_dev", "keyword_backend"
    if any(token in lowered for token in ["devops", "cloud", "sre", "site reliability", "system administrator", "sysadmin", "network", "security", "cyber", "infrastructure"]):
        return "sysadmin_devops_network", "keyword_infra"
    if any(token in lowered for token in ["product", "project", "business analyst", "pmo"]):
        return "project_product", "keyword_product"
    if any(token in lowered for token in ["sales", "account", "advocate", "customer success", "business development"]):
        return "sales_account", "keyword_sales"
    if any(token in lowered for token in ["support", "helpdesk", "qa", "testing", "test engineer"]):
        return "tech_support_helpdesk", "keyword_support"
    if any(token in lowered for token in ["manager", "director", "lead", "principal engineer", "head"]):
        return "it_governance_leadership", "keyword_leadership"
    if any(token in lowered for token in ["data", "database", "machine learning", "ai", "ml", "sap", "blockchain", "analyst"]):
        return "technical_specialized", "keyword_specialized"
    return "generic_it_ops", "fallback_generic"

normalized = df["raw_category"].apply(normalize_category)
df["supercategory"] = normalized.apply(lambda x: x[0])
df["mapping_method"] = normalized.apply(lambda x: x[1])

mapping_review = (
    df.groupby(["raw_category", "supercategory", "mapping_method"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(["count", "raw_category"], ascending=[False, True])
)
method_summary = (
    df.groupby("mapping_method")
    .size()
    .reset_index(name="rows")
    .sort_values("rows", ascending=False)
)
label_mapping_df = (
    df[["raw_category", "supercategory"]]
    .drop_duplicates()
    .rename(columns={"raw_category": "label"})
    .sort_values(["supercategory", "label"])
)

mapping_review.to_csv(RESULTS_DIR / "32_english_category_mapping_review.csv", index=False)
method_summary.to_csv(RESULTS_DIR / "32_english_mapping_method_summary.csv", index=False)
label_mapping_df.to_csv(PROCESSED_DIR / "label_to_supercategory_v1.csv", index=False)

print(df["supercategory"].value_counts())
display(method_summary)
display(mapping_review.head(30))


In [ ]:
CITY_PATTERN = re.compile(
    r"\b("
    r"new york|san francisco|los angeles|seattle|austin|chicago|boston|atlanta|dallas|houston|"
    r"london|berlin|paris|madrid|toronto|vancouver|sydney|melbourne|singapore|dubai|"
    r"bangalore|bengaluru|mumbai|delhi|hyderabad|pune"
    r")\b",
    flags=re.IGNORECASE,
)
SWAP_CITIES = ["new york", "san francisco", "london", "berlin", "singapore"]

def extract_city_mentions(text: str) -> list[str]:
    if not text:
        return []
    return sorted({m.group(0).lower() for m in CITY_PATTERN.finditer(text)})

def swap_cities_in_text(text: str, target_city: str) -> str:
    if not isinstance(text, str) or not text:
        return text

    def repl(match):
        original = match.group(0)
        if original.isupper():
            return target_city.upper()
        if original[:1].isupper():
            return target_city.title()
        return target_city.lower()

    return CITY_PATTERN.sub(repl, text)

df["city_mentions"] = df["resume_text"].apply(extract_city_mentions)
df["n_city_mentions"] = df["city_mentions"].apply(len)
df["has_city_mention"] = df["n_city_mentions"] > 0
df["source_city"] = df["city_mentions"].apply(lambda xs: xs[0] if xs else None)

location_summary = pd.DataFrame([{
    "rows_total": len(df),
    "rows_with_city_mentions": int(df["has_city_mention"].sum()),
    "share_with_city_mentions": float(df["has_city_mention"].mean()) if len(df) else 0.0,
    "median_words": float(df["text_len_words"].median()) if len(df) else 0.0,
}])
city_counts = (
    df.explode("city_mentions")
    .dropna(subset=["city_mentions"])
    .groupby("city_mentions")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

english_eval = df[df["source_city"].notna()].copy()
base_eval_df = english_eval[["resume_text", "supercategory", "raw_category", "source", "source_city"]].rename(columns={"supercategory": "label"})

counterfactual_rows = []
for _, row in english_eval.iterrows():
    for swap_city in SWAP_CITIES:
        if row["source_city"] == swap_city:
            continue
        counterfactual_rows.append({
            "row_id": int(row.name),
            "label": row["supercategory"],
            "raw_category": row["raw_category"],
            "source_city": row["source_city"],
            "swap_city": swap_city,
            "original_text": row["resume_text"],
            "swapped_text": swap_cities_in_text(row["resume_text"], swap_city),
        })

counterfactual_df = pd.DataFrame(counterfactual_rows)

location_summary.to_csv(RESULTS_DIR / "32_english_location_summary.csv", index=False)
city_counts.to_csv(RESULTS_DIR / "32_english_city_counts.csv", index=False)
base_eval_df.to_csv(RESULTS_DIR / "32_english_base_eval_slice.csv", index=False)
counterfactual_df.to_csv(RESULTS_DIR / "32_english_city_swap_counterfactuals.csv", index=False)

print("Base eval rows:", len(base_eval_df))
print("Counterfactual rows:", len(counterfactual_df))
display(location_summary)
display(city_counts.head(20))


In [ ]:
model_df = df[[
    "resume_text",
    "raw_category",
    "supercategory",
    "source",
    "raw_location",
    "source_city",
    "has_city_mention",
    "text_len_words",
    "mapping_method",
]].copy()
model_df = model_df.rename(columns={"raw_category": "label"})

class_counts = model_df["supercategory"].value_counts()
keep_classes = class_counts[class_counts >= 10].index
model_df = model_df[model_df["supercategory"].isin(keep_classes)].reset_index(drop=True)

train_df, temp_df = train_test_split(
    model_df,
    test_size=0.30,
    random_state=42,
    stratify=model_df["supercategory"],
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["supercategory"],
)

for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    split_df.to_csv(PROCESSED_DIR / f"{split_name}.csv", index=False)

combined_df = pd.concat([
    train_df.assign(split="train"),
    val_df.assign(split="val"),
    test_df.assign(split="test"),
], ignore_index=True)
combined_df.to_parquet(PROCESSED_DIR / "df_final.parquet", index=False)

split_summary = pd.DataFrame([
    {"split": "train", "rows": len(train_df)},
    {"split": "val", "rows": len(val_df)},
    {"split": "test", "rows": len(test_df)},
])
class_summary = (
    combined_df.groupby(["split", "supercategory"])
    .size()
    .reset_index(name="rows")
    .sort_values(["split", "rows"], ascending=[True, False])
)

split_summary.to_csv(RESULTS_DIR / "32_english_split_summary.csv", index=False)
class_summary.to_csv(RESULTS_DIR / "32_english_split_class_summary.csv", index=False)
combined_df.head(3)


In [ ]:
quick_answer = {
    "dataset_ready_for_next_step": True,
    "recommended_next_move": "run the strongest existing checkpoints on the English test split and the English city-swap counterfactual slice",
    "processed_dir": str(PROCESSED_DIR),
    "results_dir": str(RESULTS_DIR),
    "figures_dir": str(FIGURES_DIR),
    "n_rows_clean": int(len(df)),
    "n_rows_model_ready": int(len(model_df)),
    "n_rows_base_eval": int(len(base_eval_df)),
    "n_rows_counterfactual": int(len(counterfactual_df)),
}
(RESULTS_DIR / "32_english_quick_answer.json").write_text(
    json.dumps(quick_answer, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(json.dumps(quick_answer, indent=2, ensure_ascii=False))


In [ ]:
if HAS_MPL and not model_df.empty:
    class_counts = model_df["supercategory"].value_counts().sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(8, 5))
    class_counts.plot(kind="barh", ax=ax, color="#4c78a8")
    ax.set_title("English dataset mapped into current 9-class space")
    ax.set_xlabel("Rows")
    ax.set_ylabel("Target class")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "32_english_target9_distribution.png", dpi=180, bbox_inches="tight")
    plt.show()

    source_counts = model_df["source"].astype(str).value_counts().head(10)
    fig, ax = plt.subplots(figsize=(8, 4))
    source_counts.sort_values().plot(kind="barh", ax=ax, color="#72b7b2")
    ax.set_title("Top English dataset sources")
    ax.set_xlabel("Rows")
    ax.set_ylabel("Source")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "32_english_source_distribution.png", dpi=180, bbox_inches="tight")
    plt.show()
else:
    print("matplotlib is not installed; skipping plots.")
